# 06 API de Prédiction

## Objectif
On va exposer notre modèle XGBoost via une **API REST** qu'on va construire avec FastAPI.
L'API permet à n'importe quelle application d'obtenir une prédiction 
de consommation énergétique en envoyant une requête HTTP.

## Endpoints
| Endpoint | Méthode | Description |
|----------|---------|-------------|
| `/` | GET | Statut de l'API |
| `/predict` | POST | Prédiction H+1 avec intervalle de confiance |
| `/predict/h24` | POST | Prédiction H+24 |
| `/health` | GET | Santé du modèle |

## Architecture de l'API

Voici ce que chaque endpoint fera :

### POST /predict
- Reçoit : datetime + valeurs récentes de consommation + météo
- Calcule : toutes les features nécessaires (lags, rolling, temporelles)
- Retourne : prédiction + intervalle de confiance [bas, haut]

### Format de réponse
```json
{
  "datetime": "2018-01-15T14:00:00",
  "prediction_mw": 35420.5,
  "confidence_interval": {
    "lower": 34723.5,
    "upper": 36117.5
  },
  "mape_expected": 0.74
}
```

In [7]:
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from pathlib import Path
import os

# ============================================================
# Initialisation
# ============================================================

app = FastAPI(
    title="Energy Forecasting API",
    description="API de prédiction de consommation énergétique PJM Est",
    version="1.0.0"
)

# Chemin flexible — fonctionne en local ET sur Docker/Render
BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "data" / "processed"

model        = joblib.load(DATA_DIR / "model_xgboost.pkl")
MARGE_IC     = 697
MAPE_ATTENDU = 0.74

# ============================================================
# Schémas de données
# ============================================================

class PredictionInput(BaseModel):
    datetime          : str
    lag_1             : float
    lag_24            : float
    lag_48            : float
    lag_168           : float
    lag_8736          : float
    rolling_mean_24h  : float
    rolling_mean_168h : float
    rolling_mean_720h : float
    rolling_std_24h   : float
    rolling_std_168h  : float
    temp              : float
    humidity          : float
    windspeed         : float
    cloudcover        : float
    is_outlier        : Optional[int] = 0

class PredictionOutput(BaseModel):
    datetime              : str
    prediction_mw         : float
    confidence_interval   : dict
    mape_expected_percent : float

# ============================================================
# Endpoints
# ============================================================

@app.get("/")
def root():
    return {
        "status"     : "online",
        "model"      : "XGBoost",
        "mape_test"  : f"{MAPE_ATTENDU}%",
        "description": "API de prédiction de consommation énergétique PJM Est"
    }

@app.get("/health")
def health():
    try:
        test = np.zeros((1, 34))
        model.predict(test)
        return {"status": "healthy", "model_loaded": True}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict", response_model=PredictionOutput)
def predict(data: PredictionInput):
    try:
        dt = pd.to_datetime(data.datetime)

        features = {
            "is_outlier"       : data.is_outlier,
            "hour"             : dt.hour,
            "dayofweek"        : dt.dayofweek,
            "month"            : dt.month,
            "quarter"          : dt.quarter,
            "year"             : dt.year,
            "dayofyear"        : dt.dayofyear,
            "weekofyear"       : dt.isocalendar()[1],
            "season"           : (dt.month % 12) // 3,
            "is_holiday"       : 0,
            "is_weekend"       : int(dt.dayofweek >= 5),
            "lag_1"            : data.lag_1,
            "lag_24"           : data.lag_24,
            "lag_48"           : data.lag_48,
            "lag_168"          : data.lag_168,
            "lag_8736"         : data.lag_8736,
            "rolling_mean_24h" : data.rolling_mean_24h,
            "rolling_mean_168h": data.rolling_mean_168h,
            "rolling_mean_720h": data.rolling_mean_720h,
            "rolling_std_24h"  : data.rolling_std_24h,
            "rolling_std_168h" : data.rolling_std_168h,
            "temp"             : data.temp,
            "humidity"         : data.humidity,
            "windspeed"        : data.windspeed,
            "cloudcover"       : data.cloudcover,
            "hour_sin"         : np.sin(2 * np.pi * dt.hour / 24),
            "hour_cos"         : np.cos(2 * np.pi * dt.hour / 24),
            "dow_sin"          : np.sin(2 * np.pi * dt.dayofweek / 7),
            "dow_cos"          : np.cos(2 * np.pi * dt.dayofweek / 7),
            "month_sin"        : np.sin(2 * np.pi * dt.month / 12),
            "month_cos"        : np.cos(2 * np.pi * dt.month / 12),
            "temp_x_hour"      : data.temp * dt.hour,
            "temp_x_season"    : data.temp * ((dt.month % 12) // 3),
            "hour_x_weekend"   : dt.hour * int(dt.dayofweek >= 5),
        }

        X    = pd.DataFrame([features])
        pred = float(model.predict(X)[0])

        return PredictionOutput(
            datetime              = data.datetime,
            prediction_mw         = round(pred, 1),
            confidence_interval   = {
                "lower": round(pred - MARGE_IC, 1),
                "upper": round(pred + MARGE_IC, 1)
            },
            mape_expected_percent = MAPE_ATTENDU
        )

    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
'''

with open('../api/main.py', 'w', encoding='utf-8') as f:
    f.write(api_code)

In [4]:
# ============================================================
# Test de l'API
# ============================================================

import requests
import json
import pandas as pd
import numpy as np

# Chargement des données
df = pd.read_csv('../data/processed/pjme_final.csv',
                 index_col='Datetime',
                 parse_dates=True)

BASE_URL = "http://127.0.0.1:8000"

# --- Test 1 : endpoint racine ---
print("=== TEST 1 : Statut de l'API ===")
response = requests.get(f"{BASE_URL}/")
print(f"Status code : {response.status_code}")
print(json.dumps(response.json(), indent=2))

# --- Test 2 : health check ---
print("\n=== TEST 2 : Health Check ===")
response = requests.get(f"{BASE_URL}/health")
print(f"Status code : {response.status_code}")
print(json.dumps(response.json(), indent=2))

# --- Test 3 : prédiction ---
print("\n=== TEST 3 : Prédiction H+1 ===")

sample = df[df.index >= '2018-01-01'].iloc[0]

payload = {
    "datetime"         : "2018-01-01T02:00:00",
    "lag_1"            : float(sample['lag_1']),
    "lag_24"           : float(sample['lag_24']),
    "lag_48"           : float(sample['lag_48']),
    "lag_168"          : float(sample['lag_168']),
    "lag_8736"         : float(sample['lag_8736']),
    "rolling_mean_24h" : float(sample['rolling_mean_24h']),
    "rolling_mean_168h": float(sample['rolling_mean_168h']),
    "rolling_mean_720h": float(sample['rolling_mean_720h']),
    "rolling_std_24h"  : float(sample['rolling_std_24h']),
    "rolling_std_168h" : float(sample['rolling_std_168h']),
    "temp"             : float(sample['temp']),
    "humidity"         : float(sample['humidity']),
    "windspeed"        : float(sample['windspeed']),
    "cloudcover"       : float(sample['cloudcover']),
    "is_outlier"       : 0
}

response = requests.post(f"{BASE_URL}/predict", json=payload)
print(f"Status code : {response.status_code}")
print(json.dumps(response.json(), indent=2))

# Comparaison avec la vraie valeur
vraie_valeur = df[df.index >= '2018-01-01'].iloc[0]['PJME']
pred = response.json()['prediction_mw']
erreur = abs(vraie_valeur - pred) / vraie_valeur * 100
print(f"\nVraie valeur : {vraie_valeur:,.0f} MW")
print(f"Prédiction   : {pred:,.0f} MW")
print(f"Erreur       : {erreur:.2f}%")

=== TEST 1 : Statut de l'API ===
Status code : 200
{
  "status": "online",
  "model": "XGBoost",
  "mape_test": "0.74%",
  "description": "API de pr\u00e9diction de consommation \u00e9nerg\u00e9tique PJM Est"
}

=== TEST 2 : Health Check ===
Status code : 200
{
  "status": "healthy",
  "model_loaded": true
}

=== TEST 3 : Prédiction H+1 ===
Status code : 200
{
  "datetime": "2018-01-01T02:00:00",
  "prediction_mw": 40210.5,
  "confidence_interval": {
    "lower": 39513.5,
    "upper": 40907.5
  },
  "mape_expected_percent": 0.74
}

Vraie valeur : 39,928 MW
Prédiction   : 40,210 MW
Erreur       : 0.71%


In [5]:
# ============================================================
# Création du Dockerfile
# ============================================================

dockerfile = '''FROM python:3.11-slim

# Répertoire de travail
WORKDIR /app

# Copie des fichiers
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY api/ ./api/
COPY data/processed/model_xgboost.pkl ./data/processed/
COPY data/processed/scaler_X.pkl ./data/processed/
COPY data/processed/scaler_y.pkl ./data/processed/

# Port exposé
EXPOSE 8000

# Lancement de l'API
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('../Dockerfile', 'w', encoding='utf-8') as f:
    f.write(dockerfile)

In [ ]:
# ============================================================
# requirements pour le déploiement
# ============================================================

requirements_api = '''fastapi
uvicorn
pandas
numpy
xgboost
scikit-learn
joblib
pydantic
python-dotenv
'''

with open('../requirements_api.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_api)